# VulnSneak — Binary Model Training
**Stage 2 of 3: Fine-tuning CodeBERT for Vulnerability Detection**

---

## Objective

Train a binary sequence classifier on top of `microsoft/codebert-base` to answer
one question for any given code snippet:

> Is this code **Safe** or **Vulnerable**?

This model acts as the first stage of the VulnSneak cascade pipeline.
Any snippet classified as Vulnerable is forwarded to the Family Classification
Model in Stage 3.

---

## Inputs and Outputs

| Item | Value |
|---|---|
| Base model | `microsoft/codebert-base` |
| Training data | `data/binary_train.jsonl` — 34,014 samples |
| Validation data | `data/binary_val.jsonl` — 4,248 samples |
| Test data | `data/binary_test.jsonl` — 4,258 samples |
| Labels | `0` = Safe, `1` = Vulnerable |
| Output | `models/binary_model/` |

---

## Primary Evaluation Metric

**Recall on the Vulnerable class.**

Missing a real vulnerability (False Negative) is more dangerous than
a false alarm (False Positive). The best checkpoint is selected based
on Vulnerable Recall on the validation set, not overall accuracy.

---

## Runtime Requirement

This notebook requires a **GPU runtime**.  
In Colab: Runtime > Change runtime type > T4 GPU (or better).


## 1. Environment Setup

Install the required libraries. This cell only needs to run once per Colab session.


In [1]:
# Install required packages
# transformers : Hugging Face model hub and training utilities
# datasets     : efficient data loading and tokenization pipelines
# scikit-learn : evaluation metrics
# accelerate   : required backend for the Trainer API

import subprocess
subprocess.run([
    "pip", "install", "-q",
    "transformers==4.40.0",
    "datasets==2.19.0",
    "scikit-learn==1.4.2",
    "accelerate==0.30.0",
    "peft==0.10.0",
], check=True)

print("Installation complete.")

Installation complete.


## 2. GPU Verification

Confirm that a GPU is available before proceeding.
Training on CPU is not recommended — a single epoch on this dataset
would take several hours without GPU acceleration.


In [2]:
import torch

if not torch.cuda.is_available():
    raise EnvironmentError(
        "No GPU detected. "
        "Go to Runtime > Change runtime type and select T4 GPU."
    )

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"Device : {gpu_name}")
print(f"Memory : {gpu_mem:.1f} GB")
print(f"CUDA   : {torch.version.cuda}")


Device : NVIDIA A100-SXM4-80GB
Memory : 85.1 GB
CUDA   : 12.8


## 3. Mount Google Drive

The dataset files and trained model checkpoints will be read from and
written to Google Drive to persist across Colab sessions.

**Expected Drive structure before running this notebook:**

```
MyDrive/
  VulnSneak/
    data/
      binary_train.jsonl
      binary_val.jsonl
      binary_test.jsonl
```


In [3]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Adjust this path if your folder is named differently
# Set PROJECT_ROOT to your working directory
# Example (Colab): PROJECT_ROOT = Path("/content/drive/MyDrive/VulnSneak")
# Example (local): PROJECT_ROOT = Path("./VulnSneak")
PROJECT_ROOT = Path("/content/drive/MyDrive/VulnSneak")  # <-- change this
DATA_DIR     = PROJECT_ROOT / "data"
MODEL_DIR    = PROJECT_ROOT / "models" / "binary_model"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Verify dataset files exist
required_files = ["binary_train.jsonl", "binary_val.jsonl", "binary_test.jsonl"]
for fname in required_files:
    fpath = DATA_DIR / fname
    assert fpath.exists(), f"File not found: {fpath}"
    print(f"Found: {fpath}")

print("\nAll dataset files verified.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found: /content/drive/MyDrive/VulnSneak/data/binary_train.jsonl
Found: /content/drive/MyDrive/VulnSneak/data/binary_val.jsonl
Found: /content/drive/MyDrive/VulnSneak/data/binary_test.jsonl

All dataset files verified.


## 4. Configuration

All hyperparameters are defined here in one place.
Modify this cell before running the rest of the notebook.

### Notes on selected values

| Parameter | Value | Reason |
|---|---|---|
| `max_length` | 256 | p95 token length in this dataset is ~203. 512 wastes memory. |
| `batch_size` | 32 | Safe default for T4 16 GB. Reduce to 16 if OOM errors occur. |
| `learning_rate` | 2e-5 | Standard range for CodeBERT fine-tuning (2e-5 to 5e-5). |
| `epochs` | 5 | Enough for convergence. Early stopping will trigger sooner if needed. |
| `warmup_ratio` | 0.1 | 10% of steps used for learning rate warm-up. |


In [4]:
# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME  = "microsoft/codebert-base"
NUM_LABELS  = 2
LABEL2ID    = {"Safe": 0, "Vulnerable": 1}
ID2LABEL    = {0: "Safe", 1: "Vulnerable"}

# ── Tokenization ──────────────────────────────────────────────────────────────
MAX_LENGTH  = 256

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE      = 64
LEARNING_RATE   = 2e-5
NUM_EPOCHS      = 5
WARMUP_RATIO    = 0.1
WEIGHT_DECAY    = 0.01
RANDOM_SEED     = 42

# ── Early stopping ────────────────────────────────────────────────────────────
EARLY_STOPPING_PATIENCE = 2   # stop if val metric does not improve for 2 evals

# ── Checkpoint selection metric ───────────────────────────────────────────────
# The best checkpoint is selected based on Vulnerable Recall on the val set.
METRIC_FOR_BEST = "eval_vulnerable_recall"

print("Configuration loaded.")
print(f"  Model      : {MODEL_NAME}")
print(f"  Max length : {MAX_LENGTH}")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Epochs     : {NUM_EPOCHS}")
print(f"  LR         : {LEARNING_RATE}")


Configuration loaded.
  Model      : microsoft/codebert-base
  Max length : 256
  Batch size : 64
  Epochs     : 5
  LR         : 2e-05


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 5. Load Datasets

Load the binary JSONL files produced by `prepare_datasets.ipynb`.

Each sample contains:
- `code`    — the code snippet (str)
- `label`   — `"Safe"` or `"Vulnerable"` (str)
- `pair_id` — original pair index for traceability (int)
- `family`  — vulnerability family for per-family analysis (str)


In [6]:
import json
from collections import Counter

def load_jsonl(filepath: str) -> list[dict]:
    samples = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                samples.append(json.loads(line))
    return samples


train_samples = load_jsonl(DATA_DIR / "binary_train.jsonl")
val_samples   = load_jsonl(DATA_DIR / "binary_val.jsonl")
test_samples  = load_jsonl(DATA_DIR / "binary_test.jsonl")


def print_distribution(name: str, samples: list[dict]) -> None:
    counts = Counter(s["label"] for s in samples)
    total  = sum(counts.values())
    print(f"  {name}")
    for label in sorted(counts):
        print(f"    {label:<15} {counts[label]:>6,}  ({counts[label]/total*100:.1f}%)")


print("Label Distribution")
print("-" * 40)
print_distribution("train", train_samples)
print_distribution("val",   val_samples)
print_distribution("test",  test_samples)


Label Distribution
----------------------------------------
  train
    Safe            17,007  (50.0%)
    Vulnerable      17,007  (50.0%)
  val
    Safe             2,124  (50.0%)
    Vulnerable       2,124  (50.0%)
  test
    Safe             2,129  (50.0%)
    Vulnerable       2,129  (50.0%)


## 6. Tokenization

Tokenize the code snippets using the CodeBERT tokenizer.

The tokenizer converts raw code text into token IDs that the model
can process. `truncation=True` ensures that snippets longer than
`max_length` are cut to fit. `padding="max_length"` pads all sequences
to a uniform length within each batch.


In [7]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Vocabulary size : {tokenizer.vocab_size:,}")


class BinaryCodeDataset(Dataset):
    """
    PyTorch Dataset for binary vulnerability classification.

    Converts a list of sample dicts into tokenized tensors
    ready for the CodeBERT model.
    """

    def __init__(self, samples: list[dict], tokenizer, max_length: int):
        self.samples    = samples
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> dict:
        sample  = self.samples[idx]
        encoded = self.tokenizer(
            sample["code"],
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids":      encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels":         torch.tensor(LABEL2ID[sample["label"]], dtype=torch.long),
        }


train_dataset = BinaryCodeDataset(train_samples, tokenizer, MAX_LENGTH)
val_dataset   = BinaryCodeDataset(val_samples,   tokenizer, MAX_LENGTH)
test_dataset  = BinaryCodeDataset(test_samples,  tokenizer, MAX_LENGTH)

print(f"\nDataset sizes")
print(f"  Train : {len(train_dataset):,}")
print(f"  Val   : {len(val_dataset):,}")
print(f"  Test  : {len(test_dataset):,}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Tokenizer loaded: microsoft/codebert-base
Vocabulary size : 50,265

Dataset sizes
  Train : 34,014
  Val   : 4,248
  Test  : 4,258


## 7. Class Weights

The binary dataset is perfectly balanced (50% Safe, 50% Vulnerable),
so class weights are equal here.

This cell is included for completeness and as a reference for the
Family Model notebook, where class imbalance is present.


In [8]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

train_labels = [LABEL2ID[s["label"]] for s in train_samples]

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=np.array(train_labels),
)

class_weight_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print("Class Weights")
print("-" * 30)
for label_id, weight in enumerate(class_weights):
    print(f"  {ID2LABEL[label_id]:<15} {weight:.4f}")


Class Weights
------------------------------
  Safe            1.0000
  Vulnerable      1.0000


## 8. Load Model

Load `microsoft/codebert-base` with a sequence classification head.

The classification head is a linear layer added on top of CodeBERT's
`[CLS]` token representation. During fine-tuning, both the CodeBERT
weights and the classification head are updated.


In [9]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

model = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model loaded: {MODEL_NAME}")
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded: microsoft/codebert-base
Total parameters     : 124,647,170
Trainable parameters : 124,647,170


## 9. Evaluation Metrics

Define a custom metric function for the Trainer.

Standard metrics reported:
- **Accuracy** — overall correctness
- **Precision (Vulnerable)** — of all predicted Vulnerable, how many are correct
- **Recall (Vulnerable)** — of all actual Vulnerable, how many are caught
- **F1 (Vulnerable)** — harmonic mean of Precision and Recall

The checkpoint selection metric is **Vulnerable Recall**, because
missing a real vulnerability is the most critical failure mode.


In [10]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)


def compute_metrics(eval_pred) -> dict:
    """
    Custom metric function for the Hugging Face Trainer.

    Returns accuracy, per-class precision/recall/F1,
    and the confusion matrix flattened for logging.
    """
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        labels=[0, 1],
        zero_division=0,
    )

    cm = confusion_matrix(labels, predictions, labels=[0, 1])

    return {
        "accuracy"           : round(accuracy, 4),
        "safe_precision"     : round(precision[0], 4),
        "safe_recall"        : round(recall[0], 4),
        "safe_f1"            : round(f1[0], 4),
        "vulnerable_precision": round(precision[1], 4),
        "vulnerable_recall"  : round(recall[1], 4),
        "vulnerable_f1"      : round(f1[1], 4),
        # Confusion matrix entries for logging
        "cm_tn"              : int(cm[0, 0]),
        "cm_fp"              : int(cm[0, 1]),
        "cm_fn"              : int(cm[1, 0]),
        "cm_tp"              : int(cm[1, 1]),
    }


print("Metric function defined.")
print(f"Checkpoint selection : {METRIC_FOR_BEST} (higher is better)")


Metric function defined.
Checkpoint selection : eval_vulnerable_recall (higher is better)


## 10. Training Arguments

Configure the Hugging Face Trainer with the hyperparameters defined
in the Configuration cell.

Key decisions:
- `evaluation_strategy = "epoch"` — evaluate on validation set after every epoch
- `save_strategy = "epoch"` — save a checkpoint after every epoch
- `load_best_model_at_end = True` — restore the best checkpoint after training
- `metric_for_best_model` — uses Vulnerable Recall for checkpoint selection


In [11]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "checkpoints"),

    # Epochs and batch size
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    # Optimizer
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,

    # Evaluation and checkpointing
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model=METRIC_FOR_BEST,
    greater_is_better=True,

    # Logging
    logging_dir=str(MODEL_DIR / "logs"),
    logging_strategy="steps",
    logging_steps=100,
    report_to="none",

    # Reproducibility
    seed=RANDOM_SEED,

    # Mixed precision — speeds up training on modern GPUs
    fp16=True,
)

print("Training arguments configured.")


Training arguments configured.


## 11. Build Trainer

Assemble the Hugging Face Trainer with the model, datasets,
training arguments, and metric function.

Early stopping is added as a callback: training will stop automatically
if the Vulnerable Recall on the validation set does not improve for
`EARLY_STOPPING_PATIENCE` consecutive evaluations.


In [12]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE
        )
    ],
)

print("Trainer ready.")
print(f"  Training samples   : {len(train_dataset):,}")
print(f"  Validation samples : {len(val_dataset):,}")
print(f"  Steps per epoch    : {len(train_dataset) // BATCH_SIZE:,}")


Trainer ready.
  Training samples   : 34,014
  Validation samples : 4,248
  Steps per epoch    : 531


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


## 12. Train

Start fine-tuning. Training progress is printed every 100 steps.

Estimated time on T4 GPU: **30 to 45 minutes** for 5 epochs.

Do not close the Colab tab during training. If the session disconnects,
the last saved checkpoint in `models/binary_model/checkpoints/` can be
used to resume.


In [13]:
import time

print("Starting training...")
print("-" * 50)

start_time = time.time()
train_result = trainer.train()
elapsed = time.time() - start_time

print("-" * 50)
print(f"Training complete in {elapsed / 60:.1f} minutes.")
print(f"  Final train loss : {train_result.training_loss:.4f}")


Starting training...
--------------------------------------------------


Epoch,Training Loss,Validation Loss,Accuracy,Safe Precision,Safe Recall,Safe F1,Vulnerable Precision,Vulnerable Recall,Vulnerable F1,Cm Tn,Cm Fp,Cm Fn,Cm Tp
1,0.170600,0.114023,0.952000,0.941600,0.963700,0.952500,0.962900,0.940200,0.951400,2047,77,127,1997
2,0.092200,0.077480,0.971500,0.960000,0.984000,0.971900,0.983600,0.959000,0.971200,2090,34,87,2037
3,0.057700,0.077968,0.976500,0.963400,0.990600,0.976800,0.990300,0.962300,0.976100,2104,20,80,2044
4,0.038300,0.066334,0.981400,0.974500,0.988700,0.981500,0.988500,0.974100,0.981300,2100,24,55,2069
5,0.017200,0.071469,0.984700,0.981300,0.988200,0.984800,0.988100,0.981200,0.984600,2099,25,40,2084


--------------------------------------------------
Training complete in 8.1 minutes.
  Final train loss : 0.1191


## 13. Save Best Model

Save the best checkpoint (selected by Vulnerable Recall) and its tokenizer
to `models/binary_model/`.

This directory will be loaded during inference in `inference.ipynb`.


In [14]:
best_model_path = MODEL_DIR / "best"
best_model_path.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(best_model_path))
tokenizer.save_pretrained(str(best_model_path))

print(f"Best model saved to: {best_model_path}")
print("Contents:")
for f in sorted(best_model_path.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<35} {size_kb:>8.1f} KB")


Best model saved to: /content/drive/MyDrive/VulnSneak/models/binary_model/best
Contents:
  config.json                              0.9 KB
  merges.txt                             445.6 KB
  model.safetensors                   486926.6 KB
  special_tokens_map.json                  0.9 KB
  tokenizer.json                        2059.5 KB
  tokenizer_config.json                    1.2 KB
  training_args.bin                        5.3 KB
  vocab.json                             779.6 KB


## 14. Evaluate on Test Set

Run the best model on the held-out test set.

**Important:** This evaluation is run only once, after all training
and hyperparameter decisions have been made. Running it multiple
times to guide decisions would constitute test set leakage.

### Interpreting the Confusion Matrix

```
                  Predicted Safe    Predicted Vulnerable
Actual Safe       TN                FP  (False Alarm)
Actual Vulnerable FN (Missed!)      TP
```

- **FN (False Negatives)** — real vulnerabilities the model missed.
  This is the most critical error for a security tool.
- **FP (False Positives)** — safe code flagged as vulnerable.
  Annoying but not dangerous.


In [15]:
import numpy as np

print("Evaluating on test set...")
test_output = trainer.predict(test_dataset)
metrics     = test_output.metrics

print("\nTest Set Results")
print("=" * 45)
print(f"  Accuracy              : {metrics['test_accuracy']:.4f}")
print()
print(f"  Safe Precision        : {metrics['test_safe_precision']:.4f}")
print(f"  Safe Recall           : {metrics['test_safe_recall']:.4f}")
print(f"  Safe F1               : {metrics['test_safe_f1']:.4f}")
print()
print(f"  Vulnerable Precision  : {metrics['test_vulnerable_precision']:.4f}")
print(f"  Vulnerable Recall     : {metrics['test_vulnerable_recall']:.4f}  <- primary metric")
print(f"  Vulnerable F1         : {metrics['test_vulnerable_f1']:.4f}")
print()
print("Confusion Matrix")
print("-" * 45)
print(f"  TN (Safe    -> Safe)  : {metrics['test_cm_tn']:>6,}")
print(f"  FP (Safe    -> Vuln)  : {metrics['test_cm_fp']:>6,}")
print(f"  FN (Vuln    -> Safe)  : {metrics['test_cm_fn']:>6,}  <- missed vulnerabilities")
print(f"  TP (Vuln    -> Vuln)  : {metrics['test_cm_tp']:>6,}")


Evaluating on test set...



Test Set Results
  Accuracy              : 0.9833

  Safe Precision        : 0.9777
  Safe Recall           : 0.9892
  Safe F1               : 0.9834

  Vulnerable Precision  : 0.9891
  Vulnerable Recall     : 0.9775  <- primary metric
  Vulnerable F1         : 0.9832

Confusion Matrix
---------------------------------------------
  TN (Safe    -> Safe)  :  2,106
  FP (Safe    -> Vuln)  :     23
  FN (Vuln    -> Safe)  :     48  <- missed vulnerabilities
  TP (Vuln    -> Vuln)  :  2,081


## 15. Per-Family Error Analysis

Inspect which vulnerability families the binary model struggles with most.

This analysis breaks down the False Negative rate per family —
i.e., which vulnerability types are most frequently missed.
A high FN rate for a specific family is a signal to investigate
whether that family is underrepresented or stylistically different
from the training distribution.


In [16]:
from collections import defaultdict

predictions = np.argmax(test_output.predictions, axis=-1)
true_labels = test_output.label_ids

# Map each test sample to its family
family_results = defaultdict(lambda: {"tp": 0, "fn": 0, "fp": 0, "tn": 0})

for idx, (pred, true) in enumerate(zip(predictions, true_labels)):
    family = test_samples[idx]["family"]
    if true == 1 and pred == 1:
        family_results[family]["tp"] += 1
    elif true == 1 and pred == 0:
        family_results[family]["fn"] += 1
    elif true == 0 and pred == 1:
        family_results[family]["fp"] += 1
    else:
        family_results[family]["tn"] += 1

print("Per-Family Binary Model Performance")
print("=" * 65)
print(f"  {'Family':<35} {'TP':>5}  {'FN':>5}  {'Recall':>7}  {'FP':>5}")
print("-" * 65)

for family in sorted(family_results):
    r = family_results[family]
    total_vuln = r["tp"] + r["fn"]
    recall = r["tp"] / total_vuln if total_vuln > 0 else 0.0
    print(f"  {family:<35} {r['tp']:>5}  {r['fn']:>5}  {recall:>7.4f}  {r['fp']:>5}")


Per-Family Binary Model Performance
  Family                                 TP     FN   Recall     FP
-----------------------------------------------------------------
  CSRF                                  264      6   0.9778      2
  Insecure Cryptography                 269     27   0.9088     10
  Insecure Deserialization              219      4   0.9821      3
  OS Command Injection                  269      1   0.9963      3
  Path Traversal                        233      2   0.9915      0
  SQL Injection                         274      4   0.9856      0
  XML Injection                         224      4   0.9825      3
  XSS                                   329      0   1.0000      2


## 16. Save Evaluation Results

Persist the test set metrics to a JSON file for reference and
inclusion in the graduation project report.


In [17]:
import json
from datetime import datetime

results = {
    "model"         : MODEL_NAME,
    "max_length"    : MAX_LENGTH,
    "batch_size"    : BATCH_SIZE,
    "learning_rate" : LEARNING_RATE,
    "num_epochs"    : NUM_EPOCHS,
    "evaluated_at"  : datetime.now().isoformat(),
    "test_metrics"  : {k: v for k, v in metrics.items()},
    "per_family"    : {
        family: dict(res) for family, res in family_results.items()
    },
}

results_path = MODEL_DIR / "test_results.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to: {results_path}")


Results saved to: /content/drive/MyDrive/VulnSneak/models/binary_model/test_results.json


## 17. Summary

Training is complete. The outputs of this notebook are:

| Output | Location |
|---|---|
| Best model weights | `models/binary_model/best/` |
| Training checkpoints | `models/binary_model/checkpoints/` |
| Evaluation results | `models/binary_model/test_results.json` |

---

## Honest Caveat

All Safe samples in this dataset are **patched versions** of vulnerable code,
not independently written safe code. This means:

- The test metrics above likely **overestimate real-world performance**.
- In production, the model will encounter natively safe code that looks
  structurally different from the patched samples it was trained on.
- This limitation should be acknowledged in the graduation presentation.

---

## Next Step

Open `train_family_model.ipynb` to train the second stage of the pipeline:
the 8-class vulnerability family classifier.
